# AI vs Real Image Detector: training notebook
Run on a **free GPU**: Colab (Runtime > Change runtime type > T4 GPU) or Kaggle (Accelerator > GPU).
Run the cells top to bottom.

## 1. Get the code
Replace the URL with your GitHub repo after you push it.

In [ ]:
!git clone https://github.com/PUT_YOUR_USERNAME_HERE/ai-vs-real-detector.git
%cd ai-vs-real-detector
!pip install -q kagglehub tensorflowjs scikit-learn matplotlib

## 2. Download CIFAKE
Upload your `kaggle.json` (Kaggle > Settings > Create New Token) when asked.

In [ ]:
from google.colab import files   # skip this cell on Kaggle
import os
f = files.upload()               # choose kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!python src/download_data.py

Copy the printed path into the next cell.

In [ ]:
import os
os.environ['DATA_DIR'] = 'PASTE_PATH_PRINTED_ABOVE'

## 3. Train the baseline CNN

In [ ]:
!python src/train.py --model baseline --epochs 15

## 4. Train the MobileNetV2 model (head, then fine-tune)

In [ ]:
!python src/train.py --model mobilenet --epochs 5 --finetune-epochs 8

## 5. Add your real-world test set
Upload 100+ real photos to `realworld_test/real/` and 100+ AI images to `realworld_test/ai/` (use the file browser on the left), then run evaluation.

In [ ]:
!python src/evaluate.py --model baseline
!python src/evaluate.py --model mobilenet

## 6. Grad-CAM examples

In [ ]:
!python src/gradcam.py --image realworld_test/ai/PUT_FILE_NAME.jpg --out report/gradcam_ai.png
!python src/gradcam.py --image realworld_test/real/PUT_FILE_NAME.jpg --out report/gradcam_real.png
from IPython.display import Image, display
display(Image('report/gradcam_ai.png')); display(Image('report/gradcam_real.png'))

## 7. Export for the website

In [ ]:
!python src/export_tfjs.py

## 8. Download the results
Download `models/`, `report/` and `web/model/` and put them in your local repo, then push to GitHub.

In [ ]:
!zip -r results.zip models report web/model
from google.colab import files
files.download('results.zip')

## Part B: improve generalization (train on a mixed dataset)
The CIFAKE-only models score ~95% on CIFAKE but ~50% on the real-world test set. This part adds high-resolution real photos (COCO) and Midjourney v6 images to the training data, plus random blur/JPEG augmentation. The real-world test images are excluded from training automatically. Keep the Part A results: the before/after comparison is the main finding of the report.

In [ ]:
!pip install -q datasets
import os
os.environ['DATA_DIR'] = 'PASTE_CIFAKE_PATH_HERE'   # same CIFAKE path as in step 2
!python src/build_mixed_data.py

Train MobileNetV2 on the mixed data. `MODEL_TAG=mixed` saves it as a separate model, so the Part A models are kept.

In [ ]:
os.environ['TRAIN_DIR'] = '/content/ai-vs-real-detector/data/mixed'
os.environ['MODEL_TAG'] = 'mixed'
!python src/train.py --model mobilenet --epochs 8 --finetune-epochs 10 --robust-aug

Evaluate on the original CIFAKE test set and the held-out real-world set. Results are saved as `report/results_mobilenet_mixed.json`.

In [ ]:
!python src/evaluate.py --model mobilenet